Instructor Guide: Databricks Medallion Silver Architecture (6-Block Pattern)
The Silver layer acts as the curated single source of truth. This architecture shifts testing left (running inline unit assertions before hitting storage) and enforces zero-downtime, idempotent storage writes using Delta Lake.

In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/silver_utils

**Cell 1: Environment Setup, User Isolation & Context Resolution**

**What We Are Doing**
We establish the workspace runtime state, verify multi-tenant catalog paths, and ensure the target Medallion schemas (bronze, silver, gold) exist in Unity Catalog before running any data tasks.

**Utilities Used & Why**

DatabricksHelpers(dbutils): Dynamically resolves current_user_email and project root paths.
Why: Prevents hardcoding individual developer emails or workspace paths. If multiple developers execute this job concurrently in f1_dev, they operate safely without overwriting each other's environments.

initialize_environment(spark, catalog_name, schemas): Issues idempotent SQL DDL (CREATE SCHEMA IF NOT EXISTS) across the Medallion layers.
Why: Ensures schema dependencies are guaranteed prior to reading or writing tables, avoiding SCHEMA_NOT_FOUND runtime failures.

**Inline Test**

Asserts that current_user_email was successfully resolved and that the source table f1_dev.bronze.ferrari_stock_raw is physically present in Unity Catalog.

In [0]:
# Cell 1: Setup, Context & Environment Initialization
db_helpers = DatabricksHelpers(dbutils)
current_user_email = db_helpers.current_user()

catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

# Ensure bronze, silver, and gold schemas exist in Unity Catalog
initialize_environment(spark, catalog_name, schemas)

source_table = f"{catalog_name}.bronze.ferrari_stock_raw"
target_table = f"{catalog_name}.silver.dim_ferrari_stock"

# --- CELL 1 TEST ---
assert current_user_email and ("@" in current_user_email or current_user_email == "root"), \
    f"TEST FAILED: Invalid user context derived: '{current_user_email}'"

assert spark.catalog.tableExists(source_table), \
    f"TEST FAILED: Source bronze table '{source_table}' does not exist in catalog."

print(f"✓ Cell 1 Passed: Executed by '{current_user_email}' on catalog '{catalog_name}'.")

Cell 2: Data Extraction & Bronze String Sanitization
What We Are Doing
We read the raw Bronze DataFrame and strip out legacy null representations (such as "\N", "NULL", "null", or trailing whitespace) across all string fields.

Utilities Used & Why
sanitize_bronze_strings(df): Programmatically iterates over every string column in the schema and converts dirty string representations into native Spark NULLs via SQL CASE WHEN logic.

Why: Raw CSV imports or external API ingested data frequently store missing values as string literals ("\N" or "NULL"). If you attempt to cast "\N" directly to an integer or double, Spark throws a fatal runtime exception (CAST_INVALID_INPUT). Sanitizing first enables safe downstream casting.

Inline Test
Iterates through all string columns in the cleaned DataFrame and asserts that the count of dirty string literals ("\N", "null") is strictly 0.

In [0]:
# Cell 2: Data Extraction & Bronze String Sanitization
df_bronze = spark.table(source_table)

# Reusable string cleaning: purges '\N', 'NULL', 'null', and empty strings across all text fields
df_sanitized = sanitize_bronze_strings(df_bronze)

# --- CELL 2 TEST ---
string_cols = [f.name for f in df_sanitized.schema.fields if f.dataType.typeName() == "string"]
for col_name in string_cols:
    dirty_count = df_sanitized.filter(
        (F.col(col_name) == r"\N") | (F.lower(F.col(col_name)) == "null")
    ).count()
    assert dirty_count == 0, f"TEST FAILED: Found {dirty_count} unhandled string literals in '{col_name}'"

print(f"✓ Cell 2 Passed: String sanitization verified across {len(string_cols)} string columns.")

Cell 3: Domain Transformations & Safe Type Casting
What We Are Doing
We apply table-specific business transformations. For Ferrari market data, this includes currency symbol removal ($ and ,), date parsing across multiple formats (MM/dd/yyyy and yyyy-MM-dd), deriving calendar attributes (day_of_week), and casting numeric metrics into standard SQL types (double, bigint).

Utilities Used & Why
Standard PySpark Functions (F.coalesce, F.to_date, F.regexp_replace, .try_cast):

Why .try_cast() over .cast(): Standard .cast() throws an exception and fails the entire pipeline if a malformed string enters a row. .try_cast() safely converts invalid inputs into NULL, allowing the pipeline to continue so that the Circuit Breaker in Cell 4 can catch and report bad records gracefully.

.dropDuplicates(["trade_date"]):

Why: Ensures staging data is deduplicated against the primary key before attempting a Delta MERGE.

Inline Test
Verifies column data types (trade_date is DATE, volume is BIGINT) and checks that derived business logic (e.g., day_of_week contains valid calendar day names) functions as expected.

In [0]:
# Cell 3: Financial Domain Transformations & Safe Casting
# Coalesce date parsing to handle multiple formats ('MM/dd/yyyy' or 'yyyy-MM-dd')
df_stock_silver = (
    df_sanitized
    .withColumn(
        "trade_date", 
        F.coalesce(
            F.to_date(F.col("Date"), "MM/dd/yyyy"),
            F.to_date(F.col("Date"), "yyyy-MM-dd")
        )
    )
    .withColumn("day_of_week", F.date_format(F.col("trade_date"), "EEEE"))
    # Strip '$' and ',' symbols safely before casting to double/long
    .withColumn("close_price", F.regexp_replace(F.col("Close/Last"), r"[\$,]", "").try_cast("double"))
    .withColumn("open_price", F.regexp_replace(F.col("Open"), r"[\$,]", "").try_cast("double"))
    .withColumn("high_price", F.regexp_replace(F.col("High"), r"[\$,]", "").try_cast("double"))
    .withColumn("low_price", F.regexp_replace(F.col("Low"), r"[\$,]", "").try_cast("double"))
    .withColumn("volume", F.col("Volume").try_cast("long"))
    .select(
        "trade_date", 
        "day_of_week", 
        "close_price", 
        "open_price", 
        "high_price", 
        "low_price", 
        "volume"
    )
    .filter(F.col("trade_date").isNotNull())
    .dropDuplicates(["trade_date"])
)

# --- CELL 3 TEST ---
sample_row = df_stock_silver.first()

# 1. Primary key and price type checks
assert dict(df_stock_silver.dtypes)["trade_date"] == "date", "TEST FAILED: trade_date cast failed."
assert dict(df_stock_silver.dtypes)["close_price"] == "double", "TEST FAILED: close_price cast failed."
assert dict(df_stock_silver.dtypes)["volume"] == "bigint", "TEST FAILED: volume cast failed."

# 2. Day of week formatting check
valid_days = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"}
assert sample_row["day_of_week"] in valid_days, f"TEST FAILED: Invalid day_of_week string '{sample_row['day_of_week']}'"

print("✓ Cell 3 Passed: Ferrari stock financial transformations and date parsing verified.")

Cell 4: Pre-Write Data Quality Circuit BreakerWhat We Are DoingThis cell acts as a strict security and quality gatekeeper. It evaluates primary key integrity, uniqueness, and domain-specific business rules in memory before writing anything to Delta Lake.Utilities Used & Whyrun_circuit_breaker(df, primary_keys, prohibited_pii):Assert Primary Keys have 0 NULL values.Assert total_rows == distinct_primary_keys (no key collisions).Assert prohibited PII columns are absent.Why: Circuit Breaker Pattern. If bad data enters the pipeline, the notebook throws an AssertionError and halts execution immediately. This prevents corrupt or duplicate data from touching production storage or invalidating downstream Gold reports.Custom Financial Rule Assertions:Assert stock prices are strictly positive ($> 0$).Assert High price is always $\ge$ Low price ($High \ge Low$).Assert Volume is non-negative ($\ge 0$).

In [0]:
# Cell 4: Pre-Write Data Quality Circuit Breaker & Financial Rules
# 1. Standard Circuit Breaker: Non-Null trade_date & Uniqueness check
run_circuit_breaker(
    df=df_stock_silver, 
    primary_keys=["trade_date"], 
    prohibited_pii=set()
)

# 2. Strict Financial Market Rules Assertions
# Prices must be strictly positive
negative_prices = df_stock_silver.filter(
    (F.col("close_price") <= 0) | 
    (F.col("open_price") <= 0) | 
    (F.col("high_price") <= 0) | 
    (F.col("low_price") <= 0)
).count()
assert negative_prices == 0, f"CIRCUIT BREAKER: Found {negative_prices} records with non-positive stock prices!"

# High price must always be >= Low price
invalid_spreads = df_stock_silver.filter(F.col("high_price") < F.col("low_price")).count()
assert invalid_spreads == 0, f"CIRCUIT BREAKER: Found {invalid_spreads} records where high_price < low_price!"

# Trading volume cannot be negative
negative_volume = df_stock_silver.filter(F.col("volume") < 0).count()
assert negative_volume == 0, f"CIRCUIT BREAKER: Found {negative_volume} records with volume < 0!"

print("✓ Cell 4 Passed: Pre-write circuit breaker and financial market domain assertions passed.")

Cell 5: Target Delta DDL & Idempotent Delta Load (SCD Type 1)
What We Are Doing
We declare the target table schema with primary key constraints in Unity Catalog, enable Delta performance features, and execute an idempotent SCD Type 1 MERGE INTO.

Utilities Used & Why
upsert_silver_delta(spark, df, target_table, primary_keys):

Sets Delta Table Properties:

'delta.enableChangeDataFeed' = 'true': Enables downstream Gold pipelines to stream only row-level updates/inserts incrementally instead of re-scanning full tables.

'delta.autoOptimize.optimizeWrite' = 'true' & 'autoCompact' = 'true': Combines small Parquet files automatically on commit, eliminating the "small files problem."

Dynamically generates SQL MERGE INTO ON conditions (target.trade_date = source.trade_date).

Why MERGE INTO over mode("overwrite"): Using .write.mode("overwrite") drops underlying files, breaking active streaming checkpoints, corrupting Change Data Feed history, and causing FileNotFoundException runtime errors for concurrent dashboard readers. MERGE INTO provides non-blocking, atomic upserts.

Inline Test
Queries the target table directly from Delta storage and asserts that target_count >= staging_count.

In [0]:
# Cell 5: Target Delta DDL Creation & Idempotent Delta Load
# 1. Ensure target Silver Financial Dimension Delta table exists with Primary Key constraint
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        trade_date DATE NOT NULL,
        day_of_week STRING,
        close_price DOUBLE,
        open_price DOUBLE,
        high_price DOUBLE,
        low_price DOUBLE,
        volume BIGINT,
        CONSTRAINT dim_ferrari_stock_pk PRIMARY KEY (trade_date)
    )
    USING DELTA
""")

# 2. Reusable Upsert: Enables CDF, Auto-Optimize, and executes SCD Type 1 MERGE
upsert_silver_delta(
    spark=spark, 
    df=df_stock_silver, 
    target_table=target_table, 
    primary_keys=["trade_date"]
)

# --- CELL 5 TEST ---
target_df = spark.table(target_table)
staging_count = df_stock_silver.count()
target_count = target_df.count()

# Confirm written count matches staging expectations
assert target_count >= staging_count, \
    f"TEST FAILED: Target table count ({target_count}) is lower than staging count ({staging_count})."

print(f"✓ Cell 5 Passed: Delta MERGE load to '{target_table}' completed and verified.")

Cell 6: Unity Catalog Metadata Governance & Job Execution Exit
What We Are Doing
We apply standardized operational tags to the table in Unity Catalog to maintain enterprise governance, data auditing, and PII lineage. Once verified, the notebook terminates with a clean success signal for orchestration engines.

Utilities Used & Why
apply_silver_tags(spark, active_catalog, target_table, current_user_email, table_type):

Executes ALTER TABLE SET TAGS for 'layer' = 'silver', 'table_type' = 'dimension', 'contains_pii' = 'false_anonymized', and 'last_updated_by' = current_user_email.

Queries information_schema.table_tags to verify tags were successfully applied.

Why: Ensures audit readiness and automated compliance discovery within Unity Catalog across all workspace environments.

dbutils.notebook.exit("SUCCESS: ..."):

Why: Signals completion status to Databricks Workflows, Airflow, or Azure Data Factory DAG orchestrators.

In [0]:
# Cell 6: Unity Catalog Metadata Governance & Exit
# Apply governance tags (setting table_type='dimension')
apply_silver_tags(
    spark=spark, 
    active_catalog=catalog_name, 
    target_table=target_table, 
    current_user_email=current_user_email, 
    table_type="dimension"
)

# Set additional domain-specific tag for financial telemetry
spark.sql(f"ALTER TABLE {target_table} SET TAGS ('domain' = 'financial')")

# Clean exit for Databricks Workflows / Jobs Orchestration
dbutils.notebook.exit(f"SUCCESS: Pipeline executed cleanly by {current_user_email} on {target_table}")

Instructor Summary Matrix

![Screenshot 2026-10-08 at 18.26.09.png](./Screenshot 2026-10-08 at 18.26.09.png "Screenshot 2026-10-08 at 18.26.09.png")